In [1]:
import pandas as pd
df = pd.read_parquet(r"sentinel-2-processed.parquet")
df

,image_name,image
0,"-113.917243,51.101323.jpg",b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...
1,"-113.917264,50.966686.jpg",b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...
2,"-113.917465,51.051057.jpg",b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...
3,"-113.917555,51.071225.jpg",b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...
4,"-113.917557,51.165343.jpg",b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...
...,...,...
89791,water_body_995.jpg,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...
89792,water_body_996.jpg,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...
89793,water_body_997.jpg,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...
89794,water_body_998.jpg,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...


In [ ]:
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\Debayan\OneDrive\Desktop\Aeris")
DATASET_DIR = PROJECT_DIR / "sentinel-2-processed"

MODEL_DIR = PROJECT_DIR / "models"
REMOTECLIP_DIR = MODEL_DIR / "RemoteCLIP"
DISTILGPT2_DIR = MODEL_DIR / "DistilGPT2"

EMBEDDING_DIR = PROJECT_DIR / "remoteclip_embeddings"
LOG_DIR = PROJECT_DIR / "logs"

for path in [
    MODEL_DIR,
    REMOTECLIP_DIR,
    DISTILGPT2_DIR,
    EMBEDDING_DIR,
    LOG_DIR
]:
    path.mkdir(parents=True, exist_ok=True)

REMOTECLIP_REPO_ID = "chendelong/RemoteCLIP"
REMOTECLIP_MODEL_NAME = "ViT-B-32"

DISTILGPT2_REPO_ID = "distilbert/distilgpt2"

TARGET_SIZE = (512, 512)

print("Project directory:", PROJECT_DIR)
print("Dataset directory:", DATASET_DIR)
print("RemoteCLIP directory:", REMOTECLIP_DIR)
print("DistilGPT2 directory:", DISTILGPT2_DIR)
print("Embedding directory:", EMBEDDING_DIR)

Project directory: C:\Users\USER\OneDrive\Desktop\Aeris
Dataset directory: C:\Users\USER\OneDrive\Desktop\Aeris\sentinel-2-processed
RemoteCLIP directory: C:\Users\USER\OneDrive\Desktop\Aeris\models\RemoteCLIP
DistilGPT2 directory: C:\Users\USER\OneDrive\Desktop\Aeris\models\DistilGPT2
Embedding directory: C:\Users\USER\OneDrive\Desktop\Aeris\remoteclip_embeddings


In [2]:
%pip install -q pandas pyarrow pillow numpy tqdm huggingface_hub open_clip_torch torch transformers

Note: you may need to restart the kernel to use updated packages.


In [11]:
from huggingface_hub import hf_hub_download

REMOTECLIP_FILE = f"RemoteCLIP-{REMOTECLIP_MODEL_NAME}.pt"

REMOTECLIP_PATH = hf_hub_download(
    repo_id=REMOTECLIP_REPO_ID,
    filename=REMOTECLIP_FILE,
    local_dir=REMOTECLIP_DIR
)

print("RemoteCLIP downloaded successfully")
print("Checkpoint:", REMOTECLIP_PATH)

RemoteCLIP downloaded successfully
Checkpoint: C:\Users\USER\OneDrive\Desktop\Aeris\models\RemoteCLIP\RemoteCLIP-ViT-B-32.pt


In [12]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "open_clip_torch"
])

print("open_clip_torch installed successfully")

open_clip_torch installed successfully


In [13]:
import open_clip
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model, _, preprocess = open_clip.create_model_and_transforms(
    REMOTECLIP_MODEL_NAME,
    pretrained=REMOTECLIP_PATH
)

tokenizer = open_clip.get_tokenizer(REMOTECLIP_MODEL_NAME)

model = model.to(device)
model.eval()

print("RemoteCLIP loaded successfully")
print("Model:", REMOTECLIP_MODEL_NAME)
print("Device:", device)

RemoteCLIP loaded successfully
Model: ViT-B-32
Device: cpu


### ========Testing========

In [15]:
from io import BytesIO
from PIL import Image
import torch

image_bytes = df.iloc[0]["image"]
image_name = df.iloc[0]["image_name"]

image = Image.open(BytesIO(image_bytes)).convert("RGB")

image_input = preprocess(image).unsqueeze(0).to(device)

with torch.no_grad():
    image_features = model.encode_image(image_input)
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)

print("Image:", image_name)
print("Image size:", image.size)
print("Embedding shape:", image_features.shape)

Image: -113.917243,51.101323.jpg
Image size: (350, 350)
Embedding shape: torch.Size([1, 512])


In [18]:
text = ["an airport with an aeroplane standing on the ground"]

text_input = tokenizer(text).to(device)

with torch.no_grad():
    text_features = model.encode_text(text_input)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

print("Text:", text[0])
print("Text embedding shape:", text_features.shape)
print("Text embedding norm:", text_features.norm().item())

Text: an airport with an aeroplane standing on the ground
Text embedding shape: torch.Size([1, 512])
Text embedding norm: 0.9999999403953552


In [19]:
similarity = image_features @ text_features.T

print("Image:", image_name)
print("Text:", text[0])
print("Cosine similarity:", similarity.item())

Image: -113.917243,51.101323.jpg
Text: an airport with an aeroplane standing on the ground
Cosine similarity: 0.2240263968706131


### ======== Main Part ========

In [21]:
import numpy as np
from io import BytesIO
from PIL import Image
from tqdm.auto import tqdm

BATCH_SIZE = 32

all_embeddings = []

for start in tqdm(range(0, len(df), BATCH_SIZE)):
    batch_df = df.iloc[start:start + BATCH_SIZE]

    images = [
        preprocess(Image.open(BytesIO(image_bytes)).convert("RGB"))
        for image_bytes in batch_df["image"]
    ]

    image_batch = torch.stack(images).to(device)

    with torch.no_grad():
        batch_embeddings = model.encode_image(image_batch)
        batch_embeddings = batch_embeddings / batch_embeddings.norm(
            dim=-1, keepdim=True
        )

    all_embeddings.append(batch_embeddings.cpu().numpy())

embeddings = np.concatenate(all_embeddings, axis=0)

embedding_path = EMBEDDING_DIR / "remoteclip_embeddings.npy"

np.save(embedding_path, embeddings)

print("Embedding generation completed")
print("Number of images:", len(embeddings))
print("Embedding shape:", embeddings.shape)
print("Saved to:", embedding_path)

100%|██████████| 2807/2807 [1:02:10<00:00,  1.33s/it]


Embedding generation completed
Number of images: 89796
Embedding shape: (89796, 512)
Saved to: C:\Users\USER\OneDrive\Desktop\Aeris\remoteclip_embeddings\remoteclip_embeddings.npy


In [22]:
import numpy as np

embeddings = np.load(EMBEDDING_DIR / "remoteclip_embeddings.npy")

print("Embeddings loaded successfully")
print("Shape:", embeddings.shape)
print("Data type:", embeddings.dtype)
print("First embedding:", embeddings[0][:10])

Embeddings loaded successfully
Shape: (89796, 512)
Data type: float32
First embedding: [-0.04298054  0.02144548 -0.02599702  0.05476522  0.06295808 -0.02994829
  0.04311737  0.00255965 -0.00483273  0.01118815]


In [23]:
import pandas as pd

embedding_index = pd.DataFrame({
    "image_name": df["image_name"].values,
    "embedding_index": range(len(df))
})

index_path = EMBEDDING_DIR / "remoteclip_index.parquet"

embedding_index.to_parquet(index_path, index=False)

print("Index saved successfully")
print("Rows:", len(embedding_index))
print("Saved to:", index_path)

Index saved successfully
Rows: 89796
Saved to: C:\Users\USER\OneDrive\Desktop\Aeris\remoteclip_embeddings\remoteclip_index.parquet


In [1]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "faiss-cpu"
])

print("FAISS installed successfully")

FAISS installed successfully


In [26]:
import faiss
import numpy as np

FAISS_PATH = EMBEDDING_DIR / "remoteclip.faiss"

embedding_matrix = np.asarray(embeddings, dtype=np.float32)

dimension = embedding_matrix.shape[1]

faiss_index = faiss.IndexFlatIP(dimension)
faiss_index.add(embedding_matrix)

print("FAISS index created")
print("Vectors:", faiss_index.ntotal)
print("Dimension:", dimension)
print("FAISS path:", FAISS_PATH)

FAISS index created
Vectors: 89796
Dimension: 512
FAISS path: C:\Users\USER\OneDrive\Desktop\Aeris\remoteclip_embeddings\remoteclip.faiss


In [28]:
faiss.write_index(faiss_index, str(FAISS_PATH))

print("FAISS index saved successfully")
print("Saved to:", FAISS_PATH)

FAISS index saved successfully
Saved to: C:\Users\USER\OneDrive\Desktop\Aeris\remoteclip_embeddings\remoteclip.faiss


In [29]:
faiss_index = faiss.read_index(str(FAISS_PATH))

embeddings = np.load(
    EMBEDDING_DIR / "remoteclip_embeddings.npy"
)

embedding_index = pd.read_parquet(
    EMBEDDING_DIR / "remoteclip_index.parquet"
)

print("FAISS vectors:", faiss_index.ntotal)
print("Embeddings:", embeddings.shape)
print("Index rows:", len(embedding_index))

FAISS vectors: 89796
Embeddings: (89796, 512)
Index rows: 89796


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

DISTILGPT2_REPO_ID = "distilbert/distilgpt2"

distilgpt2_tokenizer = AutoTokenizer.from_pretrained(
    DISTILGPT2_REPO_ID,
    cache_dir=str(DISTILGPT2_DIR)
)

distilgpt2_model = AutoModelForCausalLM.from_pretrained(
    DISTILGPT2_REPO_ID,
    cache_dir=str(DISTILGPT2_DIR)
)

C:\Users\USER\AppData\Roaming\Python\Python314\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\OneDrive\Desktop\Aeris\models\DistilGPT2\models--distilbert--distilgpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
